# Demo: the full pipeline on a real traffic video

Runs the app's real pipeline (`traffic/pipeline.py`: edge-case handling → YOLOv8 → tracking → per-lane density and clearance time → signal controller → SQLite) on a public traffic video and saves:

* annotated frames and an annotated video
* a chart of vehicles per lane over time, read back from the SQLite database
* `summary.json` with counts, density time, speed and signal decisions

Video: [Traffic Flow In The Highway](https://www.pexels.com/video/traffic-flow-in-the-highway-2103099/) from Pexels (free to use under the Pexels licence). Frames are resized to 1280 px wide.

**Kaggle settings:** Internet → **On**. A GPU is optional (CPU works, just slower).

In [ ]:
!pip install -q ultralytics
%cd /tmp
!rm -rf smart-traffic-management && git clone -q https://github.com/pavanadityav-ai/smart-traffic-management.git
%cd /tmp/smart-traffic-management
!wget -q -U "Mozilla/5.0" https://videos.pexels.com/video-files/2103099/2103099-uhd_2560_1440_30fps.mp4 -O /tmp/highway.mp4
!ls -la /tmp/highway.mp4

In [ ]:
import json, os, sys, time
import cv2
sys.path.insert(0, ".")
from traffic.detection import YoloDetector
from traffic.pipeline import TrafficPipeline
from traffic.storage import Storage

VIDEO, OUT, LANES, MAX_FRAMES, WIDTH = "/tmp/highway.mp4", "/kaggle/working/demo", 4, 900, 1280
os.makedirs(OUT, exist_ok=True)
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
src_w, src_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
w, h = WIDTH, int(src_h * WIDTH / src_w)
total = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), MAX_FRAMES)
print(f"{w}x{h}, {fps:.1f} fps, processing {total} frames")

db = Storage(f"{OUT}/demo.db")
run = db.start_run("highway.mp4", "yolov8s", LANES)
pipe = TrafficPipeline(YoloDetector("yolov8s.pt", 0.4), n_lanes=LANES, storage=db, run_id=run)
writer = cv2.VideoWriter(f"{OUT}/annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
snap_at = {int(total * f) for f in (0.25, 0.5, 0.75)}
frame_idx, start, states = 0, time.time(), []
while frame_idx < total:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1
    frame = cv2.resize(frame, (w, h))
    annotated, state = pipe.process(frame, frame_idx / fps)
    writer.write(annotated)
    if frame_idx in snap_at:
        cv2.imwrite(f"{OUT}/frame_{frame_idx:05d}.jpg", annotated)
        states.append(state)
writer.release()
elapsed = time.time() - start
print(f"done: {frame_idx} frames in {elapsed:.0f} s ({frame_idx / elapsed:.1f} FPS incl. everything)")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

hist = pd.DataFrame(db.history(run))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for lane, g in hist.groupby("lane"):
    axes[0].plot(g.video_time, g["count"], label=f"Lane {lane}")
    axes[1].plot(g.video_time, g.clearance_s, label=f"Lane {lane}")
axes[0].set_title("Vehicles per lane (from SQLite)"); axes[0].set_xlabel("video time (s)")
axes[1].set_title("Estimated clearance time per lane (s)"); axes[1].set_xlabel("video time (s)")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/lanes_over_time.png", dpi=130); plt.show()

summary = {
    "video": "Pexels 2103099, Traffic Flow In The Highway (Pexels licence)",
    "resolution": f"{w}x{h} (resized from {src_w}x{src_h})", "frames": frame_idx, "video_seconds": round(frame_idx / fps, 1),
    "model": "yolov8s", "device": "GPU" if __import__("torch").cuda.is_available() else "CPU",
    "avg_inference_ms": round(pipe.total_ms / pipe.frames, 1),
    "unique_vehicles": pipe.tracker.unique_counts,
    "density_seconds": db.summary(run)["density_seconds"],
    "lanes": db.summary(run)["lanes"],
    "snapshots": [{"time_s": s["time_s"], "density": s["analysis"]["density"], "clearance_s": s["analysis"]["clearance_s"],
                   "signal": s["signal"]["phase"], "next_green_s": s["next_timing"]["green_s"],
                   "flags": [k for k, v in s["flags"].items() if v is True]} for s in states],
}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir(OUT)):
    if f.endswith(".jpg"):
        display(Image(f"{OUT}/{f}", width=900))

## Output
Everything is in `/kaggle/working/demo/` (Output tab): annotated frames, `annotated.mp4`, `lanes_over_time.png`, `summary.json` and the SQLite database.